In [1]:


from urllib import request, error
from typing import List, Dict, Any, Iterable, Tuple, Optional
import os, json, time, hashlib, requests
from tqdm import tqdm

from pathlib import Path
import json, random, itertools, gc, pathlib, statistics as st
import seqeval
import re
import requests
import time
from requests.exceptions import ReadTimeout, ConnectionError

import numpy as np
import pandas as pd
from datasets import Dataset, DatasetDict
from sklearn.model_selection import KFold, train_test_split
from collections import Counter
from sentence_transformers import SentenceTransformer
from sklearn.neighbors import BallTree
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity, pairwise_distances
from sklearn.cluster import KMeans
from IPython.display import clear_output

import torch
from transformers import (
    AutoTokenizer,
        AutoModelForTokenClassification,
    TrainingArguments,
    Trainer,
    DataCollatorForTokenClassification
)

from requests.exceptions import (
    ReadTimeout, ConnectTimeout, ConnectionError, Timeout, HTTPError
)
from seqeval.metrics import classification_report, f1_score, precision_score, recall_score, accuracy_score
from seqeval.scheme import IOB2
from evaluate import load as load_metric
from tqdm.auto import tqdm

c:\Users\user\miniconda3\envs\lora-gpu\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
HOST = "http://127.0.0.1:11434"

# >>> Escolha um que REALMENTE está instalado (copie/cole da sua /api/tags):
MODEL_KEY = "deepseek-r1:7b"
# Alternativas que você tem: "qwen2.5:14b-instruct", "deepseek-r1:14b"

# Catálogo só para opções (não interfere no nome/tag do modelo)
OLLAMA_MODELS = {
    # --- já instalados ---
    "gemma2:9b":              {"options": {"num_ctx": 8000,  "temperature": 0.2}},
    "llama3.1:8b":            {"options": {"num_ctx": 8000,  "temperature": 0.2}},
    "qwen2.5:14b-instruct":   {"options": {"num_ctx": 12000, "temperature": 0.2}},
    "deepseek-r1:14b":        {"options": {"num_ctx": 8000,  "temperature": 0.2}},

    # --- Qwen mais leve ---
    "qwen2.5:7b":             {"options": {"num_ctx": 8000,  "temperature": 0.2}},
    "qwen2.5:7b-instruct":    {"options": {"num_ctx": 8000,  "temperature": 0.2}},  # use esta p/ chat

    # --- DeepSeek R1 mais leve ---
    "deepseek-r1:7b":         {"options": {"num_ctx": 8000,  "temperature": 0.2}},
    "deepseek-r1:1.5b":       {"options": {"num_ctx": 4096,  "temperature": 0.2}},  # ultraleve

    # --- “GPT OS” (gpt-oss open-weights) ---
    "gpt-oss:20b":            {"options": {"num_ctx": 8000,  "temperature": 0.2}},
}

In [3]:
SAFE_MODEL  = re.sub(r'[^A-Za-z0-9._-]+', '_', str(MODEL_KEY))  # "gemma2:9b" -> "gemma2_9b"

# --- pastas por modelo ---
BASE_DIR    = Path("caches") / SAFE_MODEL
DIR_CACHE   = BASE_DIR / "cache"
DIR_CKPT    = BASE_DIR / "checkpoints"
DIR_LOGS    = BASE_DIR / "logs"
for d in (DIR_CACHE, DIR_CKPT, DIR_LOGS):
    d.mkdir(parents=True, exist_ok=True)

In [4]:
CHECKPOINT_DIR = ".ckpt"        # um arquivo por (modelo, split)
pathlib.Path(CHECKPOINT_DIR).mkdir(parents=True, exist_ok=True) 

In [5]:
CACHE_PATH  = DIR_CACHE / "ner_cache.jsonl" 

In [6]:
OLLAMA_MODELS[MODEL_KEY]["options"].update({
    "temperature": 0.0,
    "repeat_penalty": 1.1,
})

In [7]:
JSON_PATH = "../../data/geocorpus-v2.json"        # ajuste se estiver noutra pasta
SEED_GLOBAL = 42
FEW_SHOT_K = 20

random.seed(SEED_GLOBAL)
np.random.seed(SEED_GLOBAL)

# ---------- ler o arquivo ----------
with open(JSON_PATH, encoding="utf-8") as f:
    raw = json.load(f)



In [8]:
records_geo = [
    {
        "sentence_id": i,
        "tokens"     : item["tokens"],
        "ner_tags"   : item["ner_tokens"],
    }
    for i, item in enumerate(raw)
]

geocorpus_full = Dataset.from_list(records_geo)

In [9]:
# lista de rótulos (ordem alfabética garante consistência entre runs)
label_list = sorted({l for sent in geocorpus_full["ner_tags"] for l in sent})
label2id   = {l: i for i, l in enumerate(label_list)}
id2label   = {i: l for l, i in label2id.items()}
NUM_LABELS = len(label_list)

In [10]:
iob_labels = (
    "O "
    "B-baciaSedimentar I-baciaSedimentar "
    "B-epoca I-epoca "
    "B-idade I-idade "
    "B-periodo I-periodo "
    "B-eon I-eon "
    "B-era I-era "
    "B-magmaticas I-magmaticas "
    "B-metamorficas I-metamorficas "
    "B-sedimentaresSiliciclasticas I-sedimentaresSiliciclasticas "
    "B-sedimentaresCarbonaticas I-sedimentaresCarbonaticas "
    "B-unidadeEstratigrafica I-unidadeEstratigrafica "
    "B-contextoGeologicoDeBacia I-contextoGeologicoDeBacia "
    "B-ambienteSedimentacao I-ambienteSedimentacao "
    "B-constituinteRochaSedimentar I-constituinteRochaSedimentar "
    "B-fosseis I-fosseis "
    "B-planctonico I-planctonico "
    "B-bentonico I-bentonico "
    "B-mineral I-mineral "
    "B-procedimentoMetodologico I-procedimentoMetodologico"
)

# Splits

In [11]:
def random_splits(
    ds: Dataset, test_size=0.2, seeds: List[int] = range(30)
) -> List[DatasetDict]:
    triples = []
    for s in seeds:
        train, dev = ds.train_test_split(test_size=test_size, seed=s).values()
        triples.append(DatasetDict(train=train, dev=dev))
    return triples

In [12]:
# def split_heur_length(ds: Dataset, top_pct: float = 0.20) -> DatasetDict:
#     lengths = np.array([len(t) for t in ds["tokens"]])
#     thr = np.percentile(lengths, 100 * (1 - top_pct))
#     mask = lengths >= thr
#     return DatasetDict(train=ds.filter(~mask), dev=ds.filter(mask))


def split_heur_length(ds: Dataset, top_pct: float = 0.20) -> DatasetDict:
    """20 % das sentenças mais longas viram conjunto de validação (dev)."""
    lengths = np.array([len(t) for t in ds["tokens"]])
    thr = np.percentile(lengths, 100 * (1 - top_pct))  
    mask = lengths >= thr  

    dev_idx = np.where(mask)[0].tolist()  # índices → list[int]
    train_idx = np.where(~mask)[0].tolist()

    return DatasetDict(
        train=ds.select(train_idx),
        dev=ds.select(dev_idx),
    )

    # Tamanho da sentenças


# def split_heur_rare(ds: Dataset, freq_thr: int = 5) -> DatasetDict:
#     freq = Counter(w.lower() for sent in ds["tokens"] for w in sent)
#     rare = {w for w, c in freq.items() if c <= freq_thr}

#     def has_rare(example):
#         return any(w.lower() in rare for w in example["tokens"])

#     return DatasetDict(
#         train=ds.filter(lambda ex: not has_rare(ex)), dev=ds.filter(has_rare)
#     )


def split_heur_rare(ds: Dataset, freq_thr: int = 5) -> DatasetDict:
    freq = Counter(w.lower() for sent in ds["tokens"] for w in sent)
    rare = {w for w, c in freq.items() if c <= freq_thr}

    keep_dev = []
    for sent in ds["tokens"]:
        print(sent)
        keep_dev.append(any(w.lower() in rare for w in sent))

    dev_idx = [i for i, x in enumerate(keep_dev) if x]
    train_idx = [i for i, x in enumerate(keep_dev) if not x]

    return DatasetDict(
        train=ds.select(train_idx),
        dev=ds.select(dev_idx),
    )

    # Raridade dos tokens

In [13]:
def split_adversarial(ds: Dataset, pct_test: float = 0.20) -> DatasetDict:
    k = int(len(ds) * pct_test)
    model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
    embeds = model.encode([" ".join(t) for t in ds["tokens"]], show_progress_bar=False)
    tree = BallTree(embeds, leaf_size=40)

    idx_train, idx_test = set(range(len(ds))), []
    # semente = ponto mais central
    seed_idx = np.argmax(np.linalg.norm(embeds - embeds.mean(0), axis=1))
    idx_train.remove(seed_idx)
    idx_test.append(seed_idx)
    print(k)
    while len(idx_test) < k:
        print(len(idx_test))
        dists, _ = tree.query(embeds[list(idx_train)], k=1, return_distance=True)
        nxt = list(idx_train)[int(np.argmax(dists))]
        idx_train.remove(nxt)
        idx_test.append(nxt)

    return DatasetDict(
        train=ds.select(sorted(idx_train)),
        dev=ds.select(sorted(idx_test)),
    )

    # Maximizando Wassertein Distance


def split_adversarial_fast(ds: Dataset, pct_test: float = 0.20) -> DatasetDict:
    """
    Farthest-Point Sampling aproximando Wasserstein – versão vetorizada.
    Seleciona pct_test (~20 %) das sentenças como conjunto 'dev'.
    """
    k = int(len(ds) * pct_test)
    model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

    embeds = model.encode(
        [" ".join(t) for t in ds["tokens"]],
        show_progress_bar=True,
        convert_to_numpy=True,
        normalize_embeddings=True,  # acelera distância euclidiana ≈ cos
    )

    n = embeds.shape[0]
    idx_all = np.arange(n)

    # 1) ponto mais "central" (norma mais distante da média)
    seed_idx = np.argmax(np.linalg.norm(embeds - embeds.mean(0), axis=1))
    selected = [seed_idx]

    # 2) vetor de distâncias mínimas a qualquer ponto já escolhido
    min_dists = np.linalg.norm(embeds - embeds[seed_idx], axis=1)

    while len(selected) < k:
        next_idx = np.argmax(min_dists)
        selected.append(next_idx)

        # atualiza min_dists com a distância ao novo ponto — tudo de uma vez
        d_new = np.linalg.norm(embeds - embeds[next_idx], axis=1)
        min_dists = np.minimum(min_dists, d_new)

    train_idx = np.setdiff1d(idx_all, selected, assume_unique=True)

    return DatasetDict(
        train=ds.select(train_idx.tolist()),
        dev=ds.select(selected),
    )

In [14]:
# def loc_split(
#     dataset: Dataset, pct_test: float = 0.20, ngram: int = 4, seed: int = 42
# ) -> DatasetDict:
#     """
#     Split baseado em baixa sobreposição léxica (4-gram Jaccard).
#     Teste = pct_test das sentenças com menor overlap em relação ao pool.
#     """
#     # 1. Texto plano por sentença
#     docs = [" ".join(toks) for toks in dataset["tokens"]]

#     # 2. Vetorizar 4-grams (binário)
#     vect = CountVectorizer(
#         analyzer="word", ngram_range=(ngram, ngram), binary=True
#     ).fit(docs)
#     X = vect.transform(docs)

#     # 3. Similaridade Jaccard aproximada com matriz binária
#     # Jaccard(A,B) = |A∩B|/|A∪B| = 1 - |AΔB|/|A∪B|
#     # Usamos: overlap = (A·Bᵀ) / (|A|+|B|-A·Bᵀ)
#     bin_counts = X.sum(axis=1).A1

#     # Para cada doc i, escolhemos vizinho + próximo (fast):
#     from sklearn.metrics.pairwise import cosine_similarity

#     # (cosine no binário ∝ |A∩B|)
#     sim = cosine_similarity(X, dense_output=False)
#     # Soma dos top-k overlaps (k=5) como score
#     k = 5
#     topk = np.zeros(len(dataset))
#     for i in range(sim.shape[0]):
#         row = sim.getrow(i).toarray()[0]
#         idx = np.argpartition(-row, range(1, k + 1))[1 : k + 1]
#         # overlap ≈ |∩|
#         inter = row[idx] * bin_counts[i]
#         uni = bin_counts[i] + bin_counts[idx] - inter
#         topk[i] = (inter / uni).mean()

#     # 4. Ordenar por overlap crescente ⇒ mais “novos” vão p/ teste
#     order = np.argsort(topk)
#     n_test = int(len(dataset) * pct_test)
#     test_idx = order[:n_test]
#     train_idx = order[n_test:]

#     return DatasetDict(
#         {"train": dataset.select(train_idx), "test": dataset.select(test_idx)}
#     )

In [15]:
def loc_split(
    dataset: Dataset,
    pct_test: float = 0.20,
    pct_val: float = 0.10,
    ngram: int = 4,
    seed: int = 42,
) -> DatasetDict:
    """
    Divide por sobreposição léxica (n-gram Jaccard).
    Frações independentes para teste e validação.
    """
    docs = [" ".join(toks) for toks in dataset["tokens"]]

    vect = CountVectorizer(
        analyzer="word", ngram_range=(ngram, ngram), binary=True
    ).fit(docs)
    X = vect.transform(docs)
    bin_counts = X.sum(axis=1).A1

    sim = cosine_similarity(X, dense_output=False)
    k = 5
    topk = np.zeros(len(dataset))
    for i in range(sim.shape[0]):
        row = sim.getrow(i).toarray()[0]
        idx = np.argpartition(-row, range(1, k + 1))[1 : k + 1]
        inter = row[idx] * bin_counts[i]
        uni = bin_counts[i] + bin_counts[idx] - inter
        topk[i] = (inter / uni).mean()

    order = np.argsort(topk)  # baixo → alto overlap
    n_test = int(len(dataset) * pct_test)
    n_val = int(len(dataset) * pct_val)

    test_idx = order[:n_test]
    val_idx = order[n_test : n_test + n_val]
    train_idx = order[n_test + n_val :]

    return DatasetDict(
        {
            "train": dataset.select(train_idx),
            "val": dataset.select(val_idx),
            "test": dataset.select(test_idx),
        }
    )

In [16]:
def semantic_cluster_split(
    dataset: Dataset,
    pct_test: float = 0.20,
    pct_val: float = 0.10,
    k: int | None = None,
    seed: int = 42,
) -> DatasetDict:
    """
    Clusters SBERT → reserva clusters distantes para test/val.
    """
    if k is None:
        k = int(np.sqrt(len(dataset)))

    sbert = SentenceTransformer(
        "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
    )
    embeddings = sbert.encode(
        [" ".join(t) for t in tqdm(dataset["tokens"])],
        batch_size=64,
        show_progress_bar=False,
    )

    km = KMeans(n_clusters=k, random_state=seed, n_init=10).fit(embeddings)
    labels = km.labels_
    centroids = km.cluster_centers_

    global_center = embeddings.mean(0, keepdims=True)
    dists = pairwise_distances(centroids, global_center).flatten()

    clusters_sorted = np.argsort(-dists)  # mais distantes primeiro
    test_clusters, val_clusters = set(), set()
    total_test = total_val = 0
    n_test = int(len(dataset) * pct_test)
    n_val = int(len(dataset) * pct_val)

    for c in clusters_sorted:
        size = np.sum(labels == c)
        if total_test < n_test:  # preenche primeiro o teste
            test_clusters.add(c)
            total_test += size
        elif total_val < n_val:  # depois a validação
            val_clusters.add(c)
            total_val += size
        if total_test >= n_test and total_val >= n_val:
            break

    test_idx = np.where([lbl in test_clusters for lbl in labels])[0]
    val_idx = np.where([lbl in val_clusters for lbl in labels])[0]
    train_idx = np.where(
        [lbl not in test_clusters and lbl not in val_clusters for lbl in labels]
    )[0]

    return DatasetDict(
        {
            "train": dataset.select(train_idx),
            "val": dataset.select(val_idx),
            "test": dataset.select(test_idx),
        }
    )

In [17]:
def difficulty_scores(dataset: Dataset) -> np.ndarray:
    length = np.array([len(tok) for tok in dataset["tokens"]], dtype=float)
    length = (length - length.mean()) / length.std()

    dens = []
    for labels in dataset["ner_tags"]:
        non_o = sum(1 for l in labels if l != "O")
        dens.append(non_o / len(labels))
    dens = np.array(dens)
    dens = (dens - dens.mean()) / dens.std()

    ent_types, freq = [], Counter()
    for labels in dataset["ner_tags"]:
        types = [l[2:] for l in labels if l != "O"]
        ent_types.append(types[0] if types else "NONE")
    freq.update(ent_types)
    rarity = np.array([1 / freq[t] for t in ent_types])
    rarity = (rarity - rarity.mean()) / rarity.std()

    return length + dens + rarity


def reverse_curriculum_split(
    dataset: Dataset, pct_test: float = 0.20, pct_val: float = 0.10, seed: int = 42
) -> DatasetDict:
    scores = difficulty_scores(dataset)
    order = np.argsort(scores)  # easy→hard
    n_test = int(len(dataset) * pct_test)
    n_val = int(len(dataset) * pct_val)

    test_idx = order[-n_test:]  # hardest
    val_idx = order[-(n_test + n_val) : -n_test]
    train_idx = order[: -(n_test + n_val)]

    rng = np.random.RandomState(seed)
    rng.shuffle(train_idx)
    rng.shuffle(val_idx)
    rng.shuffle(test_idx)

    return DatasetDict(
        {
            "train": dataset.select(train_idx),
            "val": dataset.select(val_idx),
            "test": dataset.select(test_idx),
        }
    )

In [18]:
def heur_len_split(dataset: Dataset,
                   pct_test: float = 0.20,
                   pct_val : float = 0.10,
                   seed: int = 42) -> DatasetDict:
    """Testa só sentenças ≥ máx(len_train)."""
    sent_lens = np.array([len(t) for t in dataset["tokens"]])
    # separação inicial train/dev (randômica estratificando por tamanho grosso)
    idx_all   = np.arange(len(dataset))
    train_idx, temp_idx = train_test_split(idx_all,
                                           test_size=pct_test + pct_val,
                                           stratify=(sent_lens//5),  # bin len
                                           random_state=seed)
    # define longo-threshold como tamanho máx. do treino
    max_train_len = sent_lens[train_idx].max()
    # test = sentenças > threshold.  Caso falte/ sobre exemplos, ajusta.
    test_idx  = [i for i in temp_idx if sent_lens[i] > max_train_len]
    resto_idx = [i for i in temp_idx if i not in test_idx]
    # completa ou reduz para atingir pct_test
    need = int(pct_test*len(dataset)) - len(test_idx)
    if need > 0:
        test_idx.extend(resto_idx[:need])
        val_idx = resto_idx[need:]
    else:
        val_keep = int(pct_val*len(dataset))
        val_idx  = resto_idx[:val_keep]
        test_idx = test_idx[: int(pct_test*len(dataset))]
    return DatasetDict({
        "train": dataset.select(train_idx),
        "val"  : dataset.select(val_idx),
        "test" : dataset.select(test_idx),
    })

# ---------- 2) Heuristic Rare-Words ----------------------------------
def heur_rare_split(dataset: Dataset,
                    pct_test: float = 0.20,
                    pct_val : float = 0.10,
                    seed: int = 42) -> DatasetDict:
    """Testa frases que contenham palavras do quintil + raro."""
    # contagem de frequência de token
    freqs = Counter(w for toks in dataset["tokens"] for w in toks)
    # define rareza: 20 % mais raras
    thresh = np.quantile(list(freqs.values()), 0.20)
    rare_set = {w for w,c in freqs.items() if c <= thresh}
    is_rare = np.array([any(w in rare_set for w in toks)
                        for toks in dataset["tokens"]])
    rare_idx   = np.where(is_rare)[0]
    common_idx = np.where(~is_rare)[0]
    # garante proporções desejadas
    n_test = int(pct_test*len(dataset))
    n_val  = int(pct_val *len(dataset))
    rng = np.random.default_rng(seed)
    test_idx = rng.choice(rare_idx, size=min(len(rare_idx), n_test),
                          replace=False)
    resto_idx = [i for i in rare_idx if i not in test_idx] + list(common_idx)
    val_idx  = rng.choice(resto_idx, size=n_val, replace=False)
    train_idx = [i for i in resto_idx if i not in val_idx]
    return DatasetDict({
        "train": dataset.select(train_idx),
        "val"  : dataset.select(val_idx),
        "test" : dataset.select(test_idx),
    })

# ---------- 3) Standard (80-10-10) -----------------------------------
def std_split(dataset: Dataset,
              pct_test: float = 0.10,
              pct_val : float = 0.10,
              seed: int = 42) -> DatasetDict:
    """Split aleatório estratificado por comprimento (PTB-like)."""
    idx = np.arange(len(dataset))
    strat = (np.array([len(t) for t in dataset["tokens"]]) // 5)
    train_idx, temp_idx = train_test_split(idx, test_size=pct_test+pct_val,
                                          stratify=strat, random_state=seed)
    val_rel = pct_val / (pct_test+pct_val)
    val_idx, test_idx = train_test_split(temp_idx, test_size=1-val_rel,
                                         stratify=strat[temp_idx],
                                         random_state=seed)
    return DatasetDict({
        "train": dataset.select(train_idx),
        "val"  : dataset.select(val_idx),
        "test" : dataset.select(test_idx),
    })

# ---------- 4) Adversarial (approx. Wasserstein) ---------------------
def adversarial_split(dataset: Dataset,
                      pct_test: float = 0.20,
                      pct_val : float = 0.10,
                      k: int = 5,
                      seed: int = 42) -> DatasetDict:
    """
    Seleciona k frases 'mais distantes' recursivamente (BallTree + W₂)
    para compor o teste, lembrando Alg.-1 de Søgaard et al.【turn6file4】.
    """
    # SBERT embed (rápido na GPU / aceitável CPU)
    sbert = SentenceTransformer(
        "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
    )
    emb = sbert.encode([" ".join(toks) for toks in dataset["tokens"]],
                       batch_size=64, show_progress_bar=False)
    # BallTree para vizinhança eficiente
    tree = BallTree(emb, leaf_size=40)
    idx_pool = set(range(len(dataset)))
    test_idx = []
    rng = np.random.default_rng(seed)
    while len(test_idx) < int(pct_test*len(dataset)):
        # amostra candidata: ponto + longe do centro
        center = emb[list(idx_pool)].mean(0, keepdims=True)
        dists, _ = tree.query(center, k=len(idx_pool))
        farthest = list(idx_pool)[int(dists.argmax())]
        # pega-se farest e seus k-NN mais próximos  ⇒ aumenta diversidade
        nn = tree.query([emb[farthest]], k=k, return_distance=False)[0]
        for j in nn:
            if j in idx_pool and len(test_idx) < int(pct_test*len(dataset)):
                test_idx.append(j)
                idx_pool.remove(j)
    # retira val
    val_size = int(pct_val*len(dataset))
    val_idx  = rng.choice(list(idx_pool), size=val_size, replace=False)
    idx_pool -= set(val_idx)
    train_idx = list(idx_pool)
    return DatasetDict({
        "train": dataset.select(train_idx),
        "val"  : dataset.select(val_idx),
        "test" : dataset.select(test_idx),
    })

In [19]:
def heur_len_split(dataset: Dataset,
                   pct_test: float = 0.20,
                   pct_val : float = 0.10,
                   seed: int = 42,
                   bin_size: int = 10) -> DatasetDict:
    """
    Teste = sentenças mais longas que max(len(train)).
    Robusto a datasets pequenos: se estratificação falhar, usa split aleatório.
    """
    rng = np.random.default_rng(seed)
    idx_all   = np.arange(len(dataset))
    sent_lens = np.array([len(t) for t in dataset["tokens"]])

    # ------ 1) tenta split estratificado por baldes -------------------------
    strat = (sent_lens // bin_size)
    try:
        train_idx, temp_idx = train_test_split(
            idx_all,
            test_size=pct_test + pct_val,
            stratify=strat,
            random_state=seed,
        )
    except ValueError:                       # classes com 1 amostra
        train_idx, temp_idx = train_test_split(
            idx_all,
            test_size=pct_test + pct_val,
            shuffle=True,
            random_state=seed,
            stratify=None,
        )

    # ------ 2) escolhe test = len > max(train) ------------------------------
    max_train_len = sent_lens[train_idx].max()
    test_idx  = [i for i in temp_idx if sent_lens[i] > max_train_len]
    resto_idx = [i for i in temp_idx if i not in test_idx]

    # garante tamanhos exatos
    n_test_desired = int(pct_test * len(dataset))
    n_val_desired  = int(pct_val  * len(dataset))

    # completa teste se ficou pequeno
    if len(test_idx) < n_test_desired:
        extra = rng.choice(resto_idx,
                           size=n_test_desired - len(test_idx),
                           replace=False)
        test_idx.extend(extra)
        resto_idx = [i for i in resto_idx if i not in extra]

    # define validação
    val_idx  = rng.choice(resto_idx, size=n_val_desired, replace=False)
    train_idx = [i for i in idx_all if i not in test_idx and i not in val_idx]

    return DatasetDict({
        "train": dataset.select(train_idx),
        "val"  : dataset.select(val_idx),
        "test" : dataset.select(test_idx),
    })

In [20]:
def std_split(dataset,
              pct_test: float = 0.10,
              pct_val : float = 0.10,
              seed: int = 42,
              bin_size: int = 5) -> DatasetDict:
    """
    Split 80-10-10 robusto.
      • Tenta estratificar por comprimento // bin_size.
      • Se houver classes com <2 amostras, recua p/ split aleatório.
    """
    idx  = np.arange(len(dataset))
    bins = (np.array([len(t) for t in dataset["tokens"]]) // bin_size)

    try:
        train_idx, temp_idx = train_test_split(
            idx,
            test_size=pct_test + pct_val,
            stratify=bins,
            random_state=seed,
        )
    except ValueError:                       # classes muito pequenas
        train_idx, temp_idx = train_test_split(
            idx,
            test_size=pct_test + pct_val,
            shuffle=True,
            random_state=seed,
            stratify=None,
        )

    # fraciona temp em val / test mantendo proporção desejada
    val_share = pct_val / (pct_test + pct_val)
    try:
        val_idx, test_idx = train_test_split(
            temp_idx,
            test_size=1 - val_share,
            stratify=bins[temp_idx],
            random_state=seed,
        )
    except ValueError:
        val_idx, test_idx = train_test_split(
            temp_idx,
            test_size=1 - val_share,
            shuffle=True,
            random_state=seed,
            stratify=None,
        )

    return DatasetDict({
        "train": dataset.select(train_idx),
        "val"  : dataset.select(val_idx),
        "test" : dataset.select(test_idx),
    })

In [21]:
def adversarial_split(dataset: Dataset,
                      pct_test: float = 0.20,
                      pct_val : float = 0.10,
                      k: int = 5,
                      seed: int = 42) -> DatasetDict:
    """
    Versão robusta: nunca “trava” antes de atingir n_test.
    Seleciona blocos de k sentenças mais distantes do centro iterativamente.
    """
    # -------------------------------- embeds -------------------------------
    sbert = SentenceTransformer(
        "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
    )
    emb = sbert.encode([" ".join(t) for t in dataset["tokens"]],
                       batch_size=64, show_progress_bar=False)
    tree = BallTree(emb, leaf_size=40)

    n_test = int(pct_test * len(dataset))
    n_val  = int(pct_val  * len(dataset))

    idx_pool = set(range(len(dataset)))
    test_idx = []
    rng = np.random.default_rng(seed)

    print(f"Selecionando {n_test} sentenças para teste…")
    step = 0
    while len(test_idx) < n_test and idx_pool:
        if step % 5 == 0:
            print(f"  {len(test_idx)} selecionadas…")
        step += 1

        pool_list = list(idx_pool)
        center = emb[pool_list].mean(0, keepdims=True)

        # distância euclidiana ao centro global restante
        dists, _ = tree.query(center, k=len(pool_list))
        farthest_local_idx = int(dists.argmax())
        farthest_global_idx = pool_list[farthest_local_idx]

        # número efetivo de vizinhos
        k_eff = min(k, len(idx_pool))
        nn = set(tree.query([emb[farthest_global_idx]],
                            k=k_eff,
                            return_distance=False)[0])

        # adiciona vizinhos ainda não selecionados
        for j in nn:
            if j in idx_pool and len(test_idx) < n_test:
                test_idx.append(j)
                idx_pool.remove(j)

        # Se nada foi adicionado (pode acontecer quando sobram <k únicos)
        if farthest_global_idx not in test_idx:
            test_idx.append(farthest_global_idx)
            idx_pool.remove(farthest_global_idx)

    # ------------------------ validação e treino ---------------------------
    val_idx = rng.choice(list(idx_pool), size=n_val, replace=False)
    idx_pool -= set(val_idx)
    train_idx = list(idx_pool)

    return DatasetDict({
        "train": dataset.select(train_idx),
        "val"  : dataset.select(val_idx),
        "test" : dataset.select(test_idx),
    })

In [22]:
standard_split = std_split(geocorpus_full)
print('std')
# random_splt = random_splits(geocorpus_full)
# print('random')
heur_len = heur_len_split(geocorpus_full)
print("heur_len")
heur_rare = heur_rare_split(geocorpus_full)
print("heur_rare")
advers = adversarial_split(geocorpus_full)
print("advs")
loc = loc_split(geocorpus_full)
print("loc")
semantic = semantic_cluster_split(geocorpus_full)
print("semantic")
reverse = reverse_curriculum_split(geocorpus_full)
print("reverse")

std
heur_len
heur_rare
Selecionando 1054 sentenças para teste…
  0 selecionadas…
  24 selecionadas…
  47 selecionadas…
  71 selecionadas…
  96 selecionadas…
  121 selecionadas…
  144 selecionadas…
  165 selecionadas…
  187 selecionadas…
  211 selecionadas…
  232 selecionadas…
  251 selecionadas…
  271 selecionadas…
  288 selecionadas…
  305 selecionadas…
  329 selecionadas…
  349 selecionadas…
  364 selecionadas…
  383 selecionadas…
  399 selecionadas…
  417 selecionadas…
  433 selecionadas…
  452 selecionadas…
  470 selecionadas…
  488 selecionadas…
  501 selecionadas…
  515 selecionadas…
  529 selecionadas…
  542 selecionadas…
  554 selecionadas…
  565 selecionadas…
  577 selecionadas…
  590 selecionadas…
  597 selecionadas…
  610 selecionadas…
  628 selecionadas…
  642 selecionadas…
  658 selecionadas…
  671 selecionadas…
  685 selecionadas…
  698 selecionadas…
  713 selecionadas…
  724 selecionadas…
  738 selecionadas…
  754 selecionadas…
  771 selecionadas…
  785 selecionadas…
  7

100%|██████████| 5272/5272 [00:00<00:00, 878625.61it/s]


semantic
reverse


# Pipeline Ollama

In [23]:
class LLMTimeout(RuntimeError):
    """A chamada à LLM excedeu o limite de tempo configurado."""
    pass


class BadModelOutput(RuntimeError):
    """A LLM retornou um formato inválido (ex.: sem 'tags' ou JSON malformado)."""
    pass

In [24]:
def _hash_key(tokens: List[str], model: str, shot_id: str) -> str:
    h = hashlib.sha256()
    h.update(("||".join(tokens)).encode("utf-8"))
    h.update(("|" + model + "|" + shot_id).encode("utf-8"))
    return h.hexdigest()

def _append_cache(path: str, k: str, tags: List[str]) -> None:
    with open(path, "a", encoding="utf-8") as f:
        f.write(json.dumps({"k": k, "tags": tags}, ensure_ascii=False) + "\n")

def _load_cache(path: str) -> Dict[str, List[str]]:
    if not os.path.exists(path): return {}
    out: Dict[str, List[str]] = {}
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            try:
                obj = json.loads(line)
                if isinstance(obj, dict) and "k" in obj and "tags" in obj:
                    out[obj["k"]] = obj["tags"]
            except Exception:
                continue
    return out

In [25]:
def get_checkpoint_path(model: str, split_name: str) -> str:
    # um arquivo por (modelo, split)
    safe_model = model.replace("/", "_").replace(":", "_")
    safe_split = split_name.replace("/", "_")
    return os.path.join(CHECKPOINT_DIR, f"{safe_model}__{safe_split}.ckpt")

def load_checkpoint(model: str, split_name: str) -> int:
    p = get_checkpoint_path(model, split_name)
    if not os.path.exists(p): return 0
    try:
        return int(open(p, "r", encoding="utf-8").read().strip() or "0")
    except Exception:
        return 0

def save_checkpoint(model: str, split_name: str, next_index: int) -> None:
    with open(get_checkpoint_path(model, split_name), "w", encoding="utf-8") as f:
        f.write(str(next_index))


In [26]:
def tokens_to_text(tokens: List[str]) -> str:
    return " ".join(tokens)

def _coerce_tags_list(x: Any) -> List[str]:
    if isinstance(x, list) and all(isinstance(t, str) for t in x):
        return x
    raise ValueError("tags não é uma lista de strings.")

def _parse_json_loose(s: str) -> Any:
    dec = json.JSONDecoder()
    i, n, vals = 0, len(s), []
    while i < n:
        while i < n and s[i] not in "[{":
            i += 1
        if i >= n: break
        try:
            v, j = dec.raw_decode(s, idx=i)
            vals.append(v); i = j
        except json.JSONDecodeError:
            i += 1
    if not vals: raise json.JSONDecodeError("Nenhum JSON detectado", s, 0)
    return vals[0] if len(vals) == 1 else vals

In [27]:
def _normalize_ner_json(obj: Any) -> List[List[str]]:
    # Aceita: {"tags":[...]} | [{"tags":[...]}...] | [["O","B-..."], ...]
    if isinstance(obj, dict):
        if "tags" in obj:
            return [ _coerce_tags_list(obj["tags"]) ]
        if "results" in obj or "data" in obj or "output" in obj:
            return _normalize_ner_json(obj.get("results") or obj.get("data") or obj.get("output"))
        # dict indexado por "0","1",...
        if obj and all(isinstance(k, str) for k in obj.keys()):
            try:
                items = []
                for k in sorted(obj.keys(), key=lambda x: int(x)):
                    v = obj[k]
                    if isinstance(v, dict) and "tags" in v:
                        items.append(_coerce_tags_list(v["tags"]))
                    elif isinstance(v, list) and all(isinstance(t, str) for t in v):
                        items.append(_coerce_tags_list(v))
                    else:
                        raise ValueError
                return items
            except Exception:
                pass
        raise BadModelOutput("Dict sem campo 'tags' reconhecível.")
    if isinstance(obj, list):
        if not obj: return []
        if all(isinstance(x, str) for x in obj):
            return [ _coerce_tags_list(obj) ]
        if all(isinstance(x, dict) and "tags" in x for x in obj):
            return [ _coerce_tags_list(x["tags"]) for x in obj ]
        if all(isinstance(x, list) and all(isinstance(t, str) for t in x) for x in obj):
            return [ _coerce_tags_list(x) for x in obj ]
        raise BadModelOutput("Lista em formato não suportado.")
    raise BadModelOutput(f"Tipo não suportado: {type(obj).__name__}")

In [28]:
def _strip_fences(s: str) -> str:
    if s.startswith("```"):
        s = re.sub(r"^```(?:json)?\s*", "", s)
        s = re.sub(r"\s*```$", "", s)
    return s.strip()

def fmt_example(ex: dict) -> str:
    toks = ex["tokens"]
    tags = ex.get("tags") or ex.get("ner_tags")
    pairs_line = " ".join(f"{i+1}-{t}" for i, t in enumerate(tags))
    return f"Tokens: {' '.join(toks)}\nSaída: {pairs_line}"

In [29]:
def build_prompt(iob_labels: List[str], few_shot: List[Dict[str, Any]], query_tokens: List[str]) -> str:
    N = len(query_tokens)
    header = (
        "Você é um anotador especialista em NER. Rotule cada token com IOB2.\n\n"
        "Use APENAS estes rótulos:\n" + ", ".join(iob_labels) + "\n\n"
        "Saída EXCLUSIVA: JSON válido com o formato {\"tags\": [\"O\"|\"B-...\"|\"I-...\", ...]} "
        f"com exatamente {N} itens e nada mais."
    )
    examples = "\n\n### Exemplos\n" + "\n\n".join(fmt_example(ex) for ex in few_shot)
    task = f"\n\n### Tarefa\nTokens: {tokens_to_text(query_tokens)}\n\n### Saída esperada\n{{\"tags\": [ ... {N} itens ... ]}}"
    return f"{header}\n{examples}\n{task}"

In [30]:
class OllamaNER:
    def __init__(
        self,
        model: str,
        iob_labels: List[str],
        few_shot_examples: List[Dict[str, Any]],
        host: str = "http://127.0.0.1:11434",
        keep_alive: str = "30m",
        options: Optional[Dict[str, Any]] = None,
        max_seconds: int = 90,
    ) -> None:
        self.model = model
        self.keep_alive = keep_alive
        self.host = host
        self.max_seconds = max_seconds
        self.options = options or {"temperature": 0, "num_ctx": 6000}
        self.sess = requests.Session()

        few_shot_block = "\n\n".join(fmt_example(ex) for ex in few_shot_examples)
        self._base_messages = [
            {
                "role": "system",
                "content": (
                    "Você é um rotulador NER. Dado um array de tokens, devolva rótulos IOB2 "
                    "exclusivamente como JSON válido. Nunca explique."
                ),
            },
            {
                "role": "user",
                "content": (
                    "Esquema IOB2 e rótulos permitidos:\n" + ", ".join(iob_labels) +
                    "\nFormato: {\"tags\": [\"O\"|\"B-...\"|\"I-...\", ...]}\n\n"
                    "Exemplos de entrada/saída (pares índice-rótulo no texto, mas a RESPOSTA deve ser JSON):\n" +
                    few_shot_block
                ),
            },
            {"role": "assistant", "content": "OK"},
        ]

    def _chat_stream(self, user_content: str) -> str:
        payload = {
            "model": self.model,
            "keep_alive": self.keep_alive,
            "stream": True,
            "messages": self._base_messages + [{"role": "user", "content": user_content}],
            "options": self.options,
        }
        # timeout=(connect, read) — read >= max_seconds garante 'kill' no servidor cliente
        start = time.time()
        with self.sess.post(
            f"{self.host}/api/chat",
            json=payload,
            headers={"Accept": "text/event-stream"},
            timeout=(5, self.max_seconds),
            stream=True,
        ) as r:
            r.raise_for_status()
            r.encoding = "utf-8"
            chunks: List[str] = []
            for line in r.iter_lines(decode_unicode=True, chunk_size=1):
                if time.time() - start > self.max_seconds:
                    # fecha conexão de propósito
                    r.close()
                    raise LLMTimeout(f"LLM excedeu {self.max_seconds}s (stream).")
                if not line:
                    continue
                data = line[5:].strip() if line.startswith("data:") else line.strip()
                try:
                    evt = json.loads(data)
                except Exception:
                    continue
                part = evt.get("message", {}).get("content", "")
                if part:
                    chunks.append(part)
                if evt.get("done"):
                    break
        return _strip_fences("".join(chunks).strip())

    def _chat_json(self, user_content: str) -> str:
        payload = {
            "model": self.model,
            "keep_alive": self.keep_alive,
            "stream": False,
            "format": "json",
            "messages": self._base_messages + [{"role": "user", "content": user_content}],
            "options": self.options,
        }
        r = self.sess.post(
            f"{self.host}/api/chat",
            json=payload,
            timeout=(5, self.max_seconds),
        )
        r.raise_for_status()
        return _strip_fences(r.json().get("message", {}).get("content", "") or "")

    def tag_one(self, tokens: List[str], iob_labels: List[str]) -> List[str]:
        prompt = build_prompt(iob_labels, [], tokens)
        # 1ª tentativa: stream (rápido); fallback: JSON rígido
        try:
            raw = self._chat_stream(prompt)
        except (LLMTimeout, ReadTimeout, Timeout, ConnectTimeout, ConnectionError):
            # fallback sem stream (ainda com timeout duro)
            raw = self._chat_json(prompt)

        if not raw:
            raise BadModelOutput("Resposta vazia.")

        try:
            parsed = json.loads(raw)
        except Exception:
            parsed = _parse_json_loose(raw)

        tags_ll = _normalize_ner_json(parsed)  # lista de listas
        if not tags_ll:
            raise BadModelOutput("Sem 'tags' na resposta.")
        tags = tags_ll[0]
        # saneamento de comprimento
        if len(tags) != len(tokens):
            if len(tags) > len(tokens):
                tags = tags[:len(tokens)]
            else:
                tags = tags + ["O"] * (len(tokens) - len(tags))
        return tags

    def tag_batch_bisect(self, batch_tokens: List[List[str]], iob_labels: List[str]):
        if not batch_tokens:
            return []
        if len(batch_tokens) == 1:
            toks = batch_tokens[0]
            try:
                return [(True, self.tag_one(toks, iob_labels))]
            except (LLMTimeout, BadModelOutput, ReadTimeout, Timeout, ConnectionError, HTTPError, json.JSONDecodeError):
                #          ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^

                return [(False, None)]
        mid = len(batch_tokens) // 2
        left  = self.tag_batch_bisect(batch_tokens[:mid], iob_labels)
        right = self.tag_batch_bisect(batch_tokens[mid:], iob_labels)
        return left + right

# Prompts

In [31]:
# few_shot = [
#     {"tokens": ["o", "maj", "é", "composto"], "tags": ["O","O","O","O"]},
#     {"tokens": ["dioritos", "e", "monzodioritos"], "tags": ["B-magmaticas","O","I-magmaticas"]},
# ]
# query_tokens = ["o","maj","é","composto","por","monzodioritos","."]
# preds = predict_labels_ollama(few_shot, query_tokens)  # usa MODEL_KEY
# preds

In [32]:
splits = [standard_split, heur_len, heur_rare]

In [33]:
splits_names = ['standard', 'heur_len', 'heur_rare']

In [34]:
MODEL_KEY

'deepseek-r1:7b'

In [35]:
model_escolhido = MODEL_KEY

In [36]:
def run_fast_inference(
    splits,                 # lista de dicts estilo datasets.DatasetDict
    splits_names,           # nomes dos splits
    model_escolhido: str,
    iob_labels: List[str],
    FEW_SHOT_K: int,
    select_few_shot,        # fn(train_ds, K) -> List[{"tokens":[...], "ner_tags":[...]}]
    SEED_GLOBAL: int = 42,
    TARGET_SUCCESSES: int = 500,
    BATCH_SIZE: int = 4,
    HOST: str = "http://127.0.0.1:11434",
) -> Tuple[Dict[str, List[Optional[List[str]]]], Dict[str, List[List[str]]], Dict[str, Any]]:
    random.seed(SEED_GLOBAL)
    cache = _load_cache(CACHE_PATH)

    predictions: Dict[str, List[Optional[List[str]]]] = {}
    golds: Dict[str, List[List[str]]] = {}
    stats: Dict[str, Any] = {}
    total_splits = len(splits_names)

    def _statuslog_path(model: str, split_name: str) -> str:
        safe_model = model.replace("/", "_").replace(":", "_")
        safe_split = split_name.replace("/", "_")
        # reaproveita a mesma pasta dos checkpoints, se existir
        ckpt_dir = ".ckpt"
        os.makedirs(ckpt_dir, exist_ok=True)
        return os.path.join(ckpt_dir, f"{safe_model}__{safe_split}.status.jsonl")

    def _append_status(model: str, split_name: str, idx: int, status: str, err: Optional[str] = None) -> None:
        path = _statuslog_path(model, split_name)
        rec = {"i": idx, "status": status, "err": err, "t": time.time()}
        with open(path, "a", encoding="utf-8") as f:
            f.write(json.dumps(rec, ensure_ascii=False) + "\n")
    # ----------------------------------------------------------------

    for split_idx, (split_ds, split_name) in enumerate(zip(splits, splits_names), start=1):
        print(f"[{split_idx}/{total_splits}] Iniciando split: {split_name}")

        train = split_ds["train"]
        test  = split_ds["test"]

        # few-shot simples: K exemplos do train
        few_shot = select_few_shot(train, FEW_SHOT_K)

        # Mantém sua assinatura atual do OllamaNER (few_shot_text/json_mode)
        ner = OllamaNER(
            model=model_escolhido,
            few_shot_examples=few_shot,
            options={
                "temperature": 0,
                "num_ctx": 3000,
                "num_predict": 3000,
            },
            iob_labels=iob_labels
        )

        N = len(test)
        preds: List[Optional[List[str]]] = [None] * N
        glds:  List[List[str]]           = [[] for _ in range(N)]
        ok_count = 0
        fail_count = 0

        # retomada por índice (não conta “quantos bons”; conta “até onde já iteramos”)
        start_idx = load_checkpoint(model_escolhido, split_name)
        print(f"[{split_name}] alvo={TARGET_SUCCESSES} | N_test={N} | checkpoint={start_idx}")

        # Preenche rapidamente os índices anteriores usando o cache (se existir)
        for i_prefill in range(start_idx):
            ex = test[i_prefill]
            glds[i_prefill] = ex["ner_tags"]
            k = _hash_key(ex["tokens"], model_escolhido, split_name)
            if k in cache and preds[i_prefill] is None:
                preds[i_prefill] = cache[k]
                ok_count += 1  # conta como sucesso já existente
                _append_status(model_escolhido, split_name, i_prefill, "cache_ok", None)

        i = start_idx
        pending_tokens: List[List[str]] = []
        pending_pos: List[int] = []

        # status em linha (sem dependência externa)
        def _status():
            print(
                f"[{split_name}] ok={ok_count}/{TARGET_SUCCESSES} | "
                f"falhas={fail_count} | i={i}/{N}",
                end="\r",
                flush=True,
            )

        while i < N and ok_count < TARGET_SUCCESSES:
            ex = test[i]
            toks = ex["tokens"]
            glds[i] = ex["ner_tags"]
            k = _hash_key(toks, model_escolhido, split_name)

            if k in cache and preds[i] is None:
                preds[i] = cache[k]
                ok_count += 1
                save_checkpoint(model_escolhido, split_name, i + 1)
                _append_status(model_escolhido, split_name, i, "cache_ok", None)
                i += 1
                _status()
                continue

            pending_tokens.append(toks)
            pending_pos.append(i)

            if len(pending_tokens) >= BATCH_SIZE:
                results = ner.tag_batch_bisect(pending_tokens, iob_labels)
                for (ok, tags), pos, toks2 in zip(results, pending_pos, pending_tokens):
                    if ok and tags is not None:
                        preds[pos] = tags
                        cache_k = _hash_key(toks2, model_escolhido, split_name)
                        cache[cache_k] = tags
                        _append_cache(CACHE_PATH, cache_k, tags)
                        ok_count += 1
                        _append_status(model_escolhido, split_name, pos, "ok", None)
                    else:
                        fail_count += 1  # falhou? não conta; seguimos adiante
                        # Sem detalhar a exceção (tag_batch_bisect encapsula). Guarda motivo genérico.
                        _append_status(model_escolhido, split_name, pos, "fail", "batch_fail_or_timeout")
                    save_checkpoint(model_escolhido, split_name, pos + 1)
                pending_tokens, pending_pos = [], []
                _status()

            i += 1

        # flush final se sobrou algo
        if pending_tokens and ok_count < TARGET_SUCCESSES:
            results = ner.tag_batch_bisect(pending_tokens, iob_labels)
            for (ok, tags), pos, toks2 in zip(results, pending_pos, pending_tokens):
                if ok and tags is not None:
                    preds[pos] = tags
                    cache_k = _hash_key(toks2, model_escolhido, split_name)
                    cache[cache_k] = tags
                    _append_cache(CACHE_PATH, cache_k, tags)
                    ok_count += 1
                    _append_status(model_escolhido, split_name, pos, "ok", None)
                else:
                    fail_count += 1
                    _append_status(model_escolhido, split_name, pos, "fail", "batch_fail_or_timeout")
                save_checkpoint(model_escolhido, split_name, pos + 1)
            _status()

        # quebra de linha e resumo final do split
        print()
        print(
            f"[{split_idx}/{total_splits}] Finalizado {split_name}: "
            f"ok={ok_count}/{TARGET_SUCCESSES}, falhas={fail_count}, "
            f"processado_ate={load_checkpoint(model_escolhido, split_name)}/{N}"
        )

        predictions[split_name] = preds
        golds[split_name] = glds
        stats[split_name] = {
            "ok": ok_count,
            "fail": fail_count,
            "target": TARGET_SUCCESSES,
            "processed_until": load_checkpoint(model_escolhido, split_name),
            "status_log": _statuslog_path(model_escolhido, split_name),
        }

    return predictions, golds, stats

In [37]:
    # ou qwen2.5:14b-instruct etc.
IOB = iob_labels.split()  # ajuste aos seus rótulos

def select_few_shot(train_ds, K: int):
    # exemplo simples e determinístico
    idxs = list(range(min(K, len(train_ds))))
    return [ {"tokens": train_ds[i]["tokens"], "ner_tags": train_ds[i]["ner_tags"]} for i in idxs ]


predictions, golds, stats = run_fast_inference(
    splits=splits,
    splits_names=splits_names,
    model_escolhido=MODEL_KEY,
    iob_labels=IOB,
    FEW_SHOT_K=8,
    select_few_shot=select_few_shot,
    SEED_GLOBAL=123,
    TARGET_SUCCESSES=500,   # ☑️ tenta fechar 500 boas; se falhar/timeout, pula e segue
    BATCH_SIZE=4,
    HOST="http://127.0.0.1:11434",
)

print(stats)

[1/3] Iniciando split: standard
[standard] alvo=500 | N_test=528 | checkpoint=528

[1/3] Finalizado standard: ok=315/500, falhas=0, processado_ate=528/528
[2/3] Iniciando split: heur_len
[heur_len] alvo=500 | N_test=1054 | checkpoint=700

[2/3] Finalizado heur_len: ok=500/500, falhas=0, processado_ate=700/1054
[3/3] Iniciando split: heur_rare
[heur_rare] alvo=500 | N_test=1054 | checkpoint=676

[3/3] Finalizado heur_rare: ok=503/500, falhas=0, processado_ate=676/1054
{'standard': {'ok': 315, 'fail': 0, 'target': 500, 'processed_until': 528, 'status_log': '.ckpt\\deepseek-r1_7b__standard.status.jsonl'}, 'heur_len': {'ok': 500, 'fail': 0, 'target': 500, 'processed_until': 700, 'status_log': '.ckpt\\deepseek-r1_7b__heur_len.status.jsonl'}, 'heur_rare': {'ok': 503, 'fail': 0, 'target': 500, 'processed_until': 676, 'status_log': '.ckpt\\deepseek-r1_7b__heur_rare.status.jsonl'}}


In [38]:
len(golds['standard'])

528

In [39]:
len(predictions['standard'])

528

In [40]:
predictions

{'standard': [None,
  ['O',
   'O',
   'O',
   'O',
   'O',
   'O',
   'O',
   'O',
   'O',
   'O',
   'O',
   'O',
   'O',
   'O',
   'O',
   'O',
   'O',
   'I-arari',
   'B-idade',
   'O',
   'I-arari',
   'I-idade',
   'O',
   'O',
   'O',
   'O',
   'O',
   'O',
   'O',
   'O',
   'O',
   'O',
   'O',
   'O',
   'O',
   'O',
   'O',
   'O',
   'O',
   'O',
   'O',
   'O',
   'O',
   'O',
   'O'],
  None,
  ['O',
   'B-idade',
   'I-idade',
   'O',
   'B-procedimentoMetodologico',
   'I-procedimentoMetodologico',
   'O',
   'B-fosseis',
   'I-fosseis',
   'O',
   'B-planctonico',
   'I-planctonico',
   'O',
   'B-mineral',
   'I-mineral',
   'O',
   'B-procedimentoMetodologico',
   'I-procedimentoMetodologico',
   'O',
   'O',
   'O',
   'O',
   'O',
   'O',
   'O',
   'O',
   'O',
   'O',
   'O',
   'O',
   'O',
   'O',
   'O',
   'O',
   'O',
   'O',
   'O',
   'O',
   'O',
   'O',
   'O',
   'O',
   'O',
   'O',
   'O',
   'O',
   'O',
   'O',
   'O',
   'O',
   'O',
   'O',
   

In [41]:
golds

{'standard': [['O',
   'O',
   'O',
   'B-baciaSedimentar',
   'I-baciaSedimentar',
   'O',
   'O',
   'O',
   'O',
   'O',
   'O',
   'O',
   'O',
   'O',
   'O',
   'O',
   'O',
   'O',
   'O',
   'O',
   'O',
   'O',
   'O',
   'O',
   'O',
   'O',
   'O',
   'O',
   'O',
   'O',
   'O',
   'O',
   'O',
   'O',
   'O'],
  ['O',
   'O',
   'O',
   'O',
   'O',
   'O',
   'O',
   'O',
   'O',
   'O',
   'O',
   'O',
   'O',
   'O',
   'O',
   'O',
   'B-idade',
   'O',
   'O',
   'O',
   'O',
   'O',
   'O',
   'O',
   'O',
   'O',
   'O',
   'O',
   'O',
   'O',
   'O',
   'O',
   'O',
   'O',
   'O',
   'O',
   'O',
   'O',
   'O',
   'O',
   'O',
   'O',
   'O',
   'O',
   'O'],
  ['O',
   'O',
   'O',
   'B-procedimentoMetodologico',
   'I-procedimentoMetodologico',
   'O',
   'O',
   'O',
   'O',
   'O',
   'O',
   'O',
   'O',
   'O',
   'O',
   'O',
   'O',
   'O',
   'O',
   'O',
   'O',
   'O',
   'O',
   'O',
   'O',
   'O',
   'O',
   'O',
   'O',
   'O',
   'O',
   'O',
  

In [42]:
from sklearn.metrics import (
    classification_report as sk_classification_report,
    accuracy_score,
    precision_recall_fscore_support,
)

In [43]:
for nome in predictions.keys():
    print(nome)

standard
heur_len
heur_rare


In [44]:
SAFE_MODEL = re.sub(r'[^A-Za-z0-9._-]+', '_', MODEL_KEY)  # "gemma2:9b" -> "gemma2_9b"
BASE_DIR   = Path("results") / SAFE_MODEL
DIR_METRICS = BASE_DIR / "metrics"
DIR_REPORTS = BASE_DIR / "reports"
DIR_PAIRS   = BASE_DIR / "pairs"      # gold/pred pareados por linha (jsonl)
DIR_RAW     = BASE_DIR / "raw"        # dumps completos (golds/predictions)

In [45]:
RUN_TAG = time.strftime("%Y%m%d-%H%M%S")

In [46]:
for d in (DIR_METRICS, DIR_REPORTS, DIR_PAIRS, DIR_RAW):
    d.mkdir(parents=True, exist_ok=True)

In [47]:
def _coerce_seq(x):
    # aceita ["O", "B-...", ...] ou [[...]] e devolve sempre lista simples
    if x is None:
        return None
    if isinstance(x, list):
        if not x:
            return []
        if all(isinstance(t, str) for t in x):
            return x
        # casos legados: [[...]]
        if isinstance(x[0], list) and all(isinstance(t, str) for t in x[0]):
            return x[0]
    raise ValueError(f"Sequência de tags em formato não reconhecido: {type(x).__name__}")

# ---------- ACHATAR APENAS NOS ÍNDICES VÁLIDOS ----------
def _flatten_token_level_aligned(golds_split, preds_split):
    """
    golds_split: List[List[str]]
    preds_split: List[Optional[List[str]]]
    Retorna:
      y_true_flat, y_pred_flat, indices_usados
    """
    y_true_flat, y_pred_flat = [], []
    indices_usados = []

    # use somente índices com pred != None
    for i, p in enumerate(preds_split):
        if p is None:
            continue
        g = golds_split[i]

        g_seq = _coerce_seq(g)
        p_seq = _coerce_seq(p)
        if g_seq is None or p_seq is None:
            continue

        n = min(len(g_seq), len(p_seq))
        if n <= 0:
            continue

        y_true_flat.extend(g_seq[:n])
        y_pred_flat.extend(p_seq[:n])
        indices_usados.append(i)

    return y_true_flat, y_pred_flat, indices_usados

# ---------- CÁLCULO DAS MÉTRICAS ----------
def compute_token_metrics_for_split(golds_split, preds_split, labels=None):
    y_true, y_pred, indices_usados = _flatten_token_level_aligned(golds_split, preds_split)

    if not y_true:
        # nada para avaliar
        empty = {
            "precision_micro": 0.0, "recall_micro": 0.0, "f1_micro": 0.0,
            "precision_macro": 0.0, "recall_macro": 0.0, "f1_macro": 0.0,
            "precision_weighted": 0.0, "recall_weighted": 0.0, "f1_weighted": 0.0,
            "accuracy": 0.0, "support": 0, "num_labels": 0
        }
        return empty, {}, ([], []), indices_usados

    # lista estável de labels (se quiser fixar pelos seus IOB, passe em labels=IOB)
    if labels is None:
        labels = sorted({*y_true, *y_pred})

    rep = sk_classification_report(
        y_true, y_pred, labels=labels, output_dict=True, zero_division=0
    )
    p_micro, r_micro, f_micro, _ = precision_recall_fscore_support(
        y_true, y_pred, labels=labels, average='micro', zero_division=0
    )
    rep.setdefault("micro avg", {})
    rep["micro avg"].update({"precision": p_micro, "recall": r_micro, "f1-score": f_micro})

    metrics = {
        "precision_micro":    rep["micro avg"]["precision"],
        "recall_micro":       rep["micro avg"]["recall"],
        "f1_micro":           rep["micro avg"]["f1-score"],
        "precision_macro":    rep["macro avg"]["precision"],
        "recall_macro":       rep["macro avg"]["recall"],
        "f1_macro":           rep["macro avg"]["f1-score"],
        "precision_weighted": rep["weighted avg"]["precision"],
        "recall_weighted":    rep["weighted avg"]["recall"],
        "f1_weighted":        rep["weighted avg"]["f1-score"],
        "accuracy":           accuracy_score(y_true, y_pred),
        "support":            len(y_true),
        "num_labels":         len(labels),
    }
    return metrics, rep, (y_true, y_pred), indices_usados


# ================= AVALIAÇÃO POR SPLIT =================
metrics_by_split = {}
reports_by_split = {}
indices_by_split = {}  # <-- novo: guarda os índices usados por split

# Se quiser uma lista fixa (estável) de labels entre splits:
LABELS_FIXOS = IOB  # ou None para inferir dos dados

for nome in predictions.keys():
    m, rep, (y_true, y_pred), idx_used = compute_token_metrics_for_split(
        golds[nome], predictions[nome], labels=LABELS_FIXOS
    )
    metrics_by_split[nome] = m
    reports_by_split[nome] = rep
    indices_by_split[nome] = idx_used

# ---- salvar por split ----
for nome in predictions.keys():
    with open(DIR_METRICS / f"{nome}.metrics.json", "w", encoding="utf-8") as f:
        json.dump(metrics_by_split[nome], f, ensure_ascii=False, indent=2)

    with open(DIR_REPORTS / f"{nome}.report.json", "w", encoding="utf-8") as f:
        json.dump(reports_by_split[nome], f, ensure_ascii=False, indent=2)

    # salva os pares só dos índices usados (garante alinhamento 1:1)
    pairs_path = DIR_PAIRS / f"{nome}.jsonl"
    with open(pairs_path, "w", encoding="utf-8") as f:
        for i in indices_by_split[nome]:
            rec = {"idx": i, "gold": golds[nome][i], "pred": predictions[nome][i]}
            f.write(json.dumps(rec, ensure_ascii=False) + "\n")

# (opcional) salvar os índices usados para auditoria
with open(DIR_REPORTS / "indices_usados.json", "w", encoding="utf-8") as f:
    json.dump(indices_by_split, f, ensure_ascii=False, indent=2)

# ---- salvar agregados (opcional) ----
with open(DIR_RAW / f"golds.{RUN_TAG}.json", "w", encoding="utf-8") as f:
    json.dump(golds, f, ensure_ascii=False)
with open(DIR_RAW / f"predictions.{RUN_TAG}.json", "w", encoding="utf-8") as f:
    json.dump(predictions, f, ensure_ascii=False)